# 03 · Configuração 3: especialização com LoRA
O LoRA ensina **comportamento** (JSON, citar o id certo, recusar quando o trecho não está no
contexto), não decora a lei. Treinado só com `avaliacao/lora_treino.jsonl` (disjunto do teste).
Para cada pergunta que consta geramos também um exemplo **sem** o trecho certo, cujo alvo é a recusa.

In [1]:
# Preparação do ambiente (Colab ou local)
import os, sys, subprocess
REPO_URL = "https://github.com/giuliano10/GS-GENENERATIVE-AI.git"
SUBPASTA = "assistente-acessibilidade-calcadas"   # pasta do projeto dentro do repositório

if "google.colab" in sys.modules:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    else:   # mesma sessão de outro notebook: atualiza o código
        subprocess.run(["git", "-C", "/content/repo", "pull", "-q"])
    os.chdir(os.path.join("/content/repo", SUBPASTA))
    # O Colab já traz torch, transformers, numpy etc. Instalamos SÓ o que falta, sem fixar
    # versões: reinstalar o torch com a sessão aberta quebra o "import torch".
    import importlib.util
    faltando = [pip for mod, pip in [("peft", "peft"), ("accelerate", "accelerate"),
                ("sentence_transformers", "sentence-transformers"), ("fastapi", "fastapi"),
                ("httpx", "httpx"), ("tabulate", "tabulate"), ("bs4", "beautifulsoup4"),
                ("lxml", "lxml")] if importlib.util.find_spec(mod) is None]
    # O torchao que vem no Colab é antigo e faz o peft recusar o import; o projeto não o usa.
    if importlib.util.find_spec("torchao") is not None:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    if faltando:
        print("Instalando:", faltando)
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *faltando],
                           capture_output=True, text=True)
        if r.returncode != 0:
            print(r.stdout[-3000:], r.stderr[-3000:])
            raise RuntimeError("Falha no pip install (detalhes acima)")
    # Cada notebook do Colab roda numa máquina separada: os resultados passam de um notebook
    # para o outro por uma pasta no Google Drive.
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = "/content/drive/MyDrive/assistente-calcadas-resultados"
    os.makedirs(PERSIST, exist_ok=True)
    subprocess.run(["cp", "-r", f"{PERSIST}/.", "."])
    print("Resultados anteriores trazidos do Drive:", sorted(os.listdir(PERSIST)))
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Pasta do projeto:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Resultados anteriores trazidos do Drive: ['avaliacao']
Pasta do projeto: /content/repo/assistente-acessibilidade-calcadas


In [2]:
# Confere se os arquivos dos notebooks anteriores chegaram
for arq in ["avaliacao/limiar.json"]:
    assert os.path.exists(arq), f"Falta {arq}: rode antes o notebook 01 até o fim (ele salva no Drive)."
print("ok")

ok


In [3]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (use Ambiente de execução > Alterar tipo > T4 GPU)")

GPU: Tesla T4


In [4]:
from src.base_conhecimento import carregar_trechos
from src.avaliacao import carregar_jsonl
from src.busca import Buscador, carregar_limiar
from src.config import ARQ_TESTE, ARQ_TREINO
from src.lora import montar_exemplos, treinar
trechos = carregar_trechos(); por_id = {t["id"]: t for t in trechos}
treino, teste = carregar_jsonl(ARQ_TREINO), carregar_jsonl(ARQ_TESTE)
buscador = Buscador(trechos, modo="hibrido")
exemplos = montar_exemplos(treino, buscador, por_id)
print(len(exemplos), "exemplos de treino")
print(exemplos[0]["mensagens"][1]["content"][:600], "\n\nALVO:", exemplos[0]["alvo"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

109 exemplos de treino
TRECHOS:
[id: SP-L16673-art1] (Lei Municipal nº 16.673/2017 (Estatuto do Pedestre – São Paulo), Art. 1)
Art. 1º Fica instituído no Município de São Paulo o Estatuto do Pedestre.
---
[id: SP-L16673-art8-p2] (Lei Municipal nº 16.673/2017 (Estatuto do Pedestre – São Paulo), Art. 8 (parte 2))
Art. 8º O Estatuto do Pedestre tem os seguintes objetivos: [...]
XII - a melhoria das condições de saúde da população pela prática da atividade física da caminhada;
XIII - a conscientização da sociedade quanto aos efeitos indesejáveis da utilização do veículo automotor nas locomoções urbanas;
XIV - o incentiv 

ALVO: {"resposta": "Não consta na base de conhecimento.", "fontes": [], "recusa": true}


In [5]:
treinar(exemplos, epocas=3, lr=2e-4, r=16)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


Step,Training Loss
5,0.341664
10,0.286485
15,0.196890
20,0.171283
25,0.179747
30,0.211769
35,0.139126
40,0.163025


Adaptador salvo em /content/repo/assistente-acessibilidade-calcadas/modelos/lora-acessibilidade


## Avaliação com o adaptador

In [6]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()
from src.gerador import carregar_modelo
from src.experimento import rodar
modelo = carregar_modelo(com_lora=True); print(modelo.nome)
rag_lora = rodar("rag_lora", teste, modelo, buscador, carregar_limiar())
for s in rag_lora[:3]: print(s["pergunta"], "\n ->", s["bruto"][:300], "\n")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Qwen/Qwen2.5-1.5B-Instruct + LoRA(/content/repo/assistente-acessibilidade-calcadas/modelos/lora-acessibilidade)
[rag_lora] 10/60
[rag_lora] 20/60
[rag_lora] 30/60
[rag_lora] 40/60
[rag_lora] 50/60
[rag_lora] 60/60
Qual a largura mínima da faixa livre da calçada em São Paulo? 
 -> {"resposta": "Deve ser de pelo menos 1,20m (um metro e vinte centímetros).", "fontes": ["SP-D59671-art26"], "recusa": false} 

Qual a inclinação transversal máxima permitida na faixa livre da calçada? 
 -> {"resposta": "Até 8,33%, com inclinação transversal máxima de 3% na faixa de acesso e 3% na faixa de serviço.", "fontes": ["SP-D59671-art22"], "recusa": false} 

Qual a largura mínima da faixa de serviço da calçada? 
 -> {"resposta": "Deve situar-se em posição adjacente à guia, exceto em situações atípicas, mediante autorização da Subprefeitura; e ter largura mínima de 70 cm.", "fontes": ["SP-D59671-art4-p2"], "recusa": false} 



O adaptador fica em `modelos/lora-acessibilidade/` (~20 MB) e é versionado no repositório.

In [7]:
# Salva os resultados no Google Drive (para o próximo notebook usar)
import shutil
FIXOS = {"casos_teste.jsonl", "lora_treino.jsonl", ".gitkeep"}
if "google.colab" in sys.modules:
    for pasta in ["avaliacao", "modelos"]:
        if os.path.exists(pasta):
            shutil.copytree(pasta, os.path.join(PERSIST, pasta), dirs_exist_ok=True,
                            ignore=lambda d, nomes: [n for n in nomes if n in FIXOS or n.endswith("_ckpt")])
    print("Salvo no Drive:", {p: sorted(os.listdir(os.path.join(PERSIST, p)))
                              for p in os.listdir(PERSIST) if os.path.isdir(os.path.join(PERSIST, p))})

Salvo no Drive: {'avaliacao': ['busca_recall.csv', 'limiar.json', 'limiar.png', 'respostas_rag.jsonl', 'respostas_rag_lora.jsonl', 'respostas_sem_contexto.jsonl'], 'modelos': ['lora-acessibilidade']}
